# Last Layer Fine-Tuning SigLIP2-SO400M — folds_v3
Training last-layer FT, simpan model terbaik ke Drive, lalu **evaluasi di data train mentah** (bukan test set) untuk melihat sisa kesalahan prediksi setelah cleaning.

In [ ]:
import os, sys, shutil
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

# 1. Mount Google Drive
drive.mount("/content/drive", force_remount=True)

# 2. Clone atau Pull Repo Terbaru
REPO_DIR = "/content/satria-data-bdcugm02"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://github.com/agaggigit/satria-data-bdcugm02.git {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

# 3. Install dependensi
os.system("pip install -q -U 'torchao>=0.16.0'")
os.system(f"pip install -q --no-warn-conflicts -r {REPO_DIR}/track_b/requirements.txt")

# 4. Copy gambar train ke /tmp untuk I/O cepat
DRIVE_TRAIN_DIR = "/content/drive/MyDrive/BDC2026/train"
LOCAL_TRAIN_DIR = "/tmp/dataset/train"

def copy_img_worker(args):
    src, dst = args
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst) or os.path.getsize(src) != os.path.getsize(dst):
        shutil.copy2(src, dst)

if os.path.exists(DRIVE_TRAIN_DIR):
    print("Memulai copy gambar ke /tmp dengan 32 workers...")
    all_imgs = [
        os.path.join(r, f)
        for r, _, fs in os.walk(DRIVE_TRAIN_DIR)
        for f in fs if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ]
    pairs = [(src, os.path.join(LOCAL_TRAIN_DIR, os.path.relpath(src, DRIVE_TRAIN_DIR))) for src in all_imgs]
    with ThreadPoolExecutor(max_workers=32) as ex:
        list(ex.map(copy_img_worker, pairs))
    print(f"Selesai: {len(pairs)} gambar dicopy ke {LOCAL_TRAIN_DIR}")
else:
    print("Folder gambar Drive tidak ditemukan.")


## Training — Last Layer (4 blok terakhir dibuka)
Model terbaik **otomatis disimpan ke Drive** setiap val_f1 meningkat.

In [ ]:
import sys, importlib
sys.path.insert(0, "/content/satria-data-bdcugm02/track_a/src")
sys.path.insert(0, "/content/satria-data-bdcugm02/track_b/src")
sys.path.insert(0, "/content/satria-data-bdcugm02/track_b/experiments")

import embed, lora_ft, config
importlib.reload(config); importlib.reload(embed); importlib.reload(lora_ft)
from config import CFG, make_cfg

VARIANT    = "last_layer"
CHECKPOINT = "google/siglip2-so400m-patch14-384"

cfg = make_cfg(
    run_name    = f"{VARIANT}_ft_fold0_5ep_v3",
    folds_csv   = CFG.folds_v3_csv,
    batch       = 8,
    accum_steps = 4,
)

# Training -- model terbaik otomatis disimpan ke Drive tiap val_f1 naik
result = lora_ft.run_smoke_test_fold0(
    variant       = VARIANT,
    cfg           = cfg,
    checkpoint    = CHECKPOINT,
    max_epochs    = 5,
    n_last_blocks = 4,
)
result


## Konfirmasi Checkpoint
Cek file model yang disimpan di Drive.

In [ ]:
import os
ckpt_path  = result["best_ckpt_path"]
best_epoch = result["best_epoch"]
best_f1    = result["best_val_f1"]

if ckpt_path and os.path.exists(ckpt_path):
    size_mb = os.path.getsize(ckpt_path) / 1e6
    print("Checkpoint tersimpan!")
    print(f"  Path       : {ckpt_path}")
    print(f"  Ukuran     : {size_mb:.1f} MB")
    print(f"  Best Epoch : {best_epoch}")
    print(f"  Best val_f1: {best_f1:.4f}")
else:
    print("Checkpoint tidak ditemukan. Cek log training di atas.")


## Evaluasi di Data Train Mentah
Jalankan inference di seluruh folder train (semua ~26K gambar).  
Tujuan: lihat berapa gambar yang **masih salah prediksi** oleh model setelah fine-tuning.  
Hasil disimpan ke `eval_train_mentah_lastlayer_v3.csv` di Drive.

In [ ]:
import importlib
importlib.reload(lora_ft)

CKPT_PATH = result["best_ckpt_path"]
# CKPT_PATH = "/content/drive/MyDrive/BDC2026apace/output_trackB/last_layer_ft_fold0_5ep_v3_best.pt"

# Pakai /tmp jika sudah dicopy, atau fallback ke Drive
TRAIN_DIR = "/tmp/dataset/train"
if not os.path.exists(TRAIN_DIR):
    TRAIN_DIR = "/content/drive/MyDrive/BDC2026/train"
    print(f"Gambar belum dicopy ke /tmp, pakai Drive langsung: {TRAIN_DIR}")

# Evaluasi di seluruh data TRAIN mentah
# data_dir diisi -> scan folder train, bukan test
filepaths, preds, probs = lora_ft.evaluate_on_test(
    ckpt_path  = CKPT_PATH,
    cfg        = cfg,
    batch_size = 32,
    data_dir   = TRAIN_DIR,   # <-- ini kuncinya: pakai train, bukan test
)


In [ ]:
import pandas as pd
import os

CLASS_NAMES = {0: "Recyclable", 1: "Electronic", 2: "Organic"}

# Cari label asli dari folds_v3 untuk dibandingkan
import pandas as pd
folds_v3 = pd.read_csv(cfg.folds_v3_csv)

# Normalisasi path: ambil nama file saja untuk join
fp_to_label = dict(zip(
    folds_v3["filepath"].apply(os.path.basename),
    folds_v3["label"]
))

df_train_eval = pd.DataFrame({
    "filepath"        : filepaths,
    "filename"        : [os.path.basename(p) for p in filepaths],
    "pred_label_id"   : preds,
    "pred_label_name" : [CLASS_NAMES[p] for p in preds],
    "prob_Recyclable" : [p[0] for p in probs],
    "prob_Electronic" : [p[1] for p in probs],
    "prob_Organic"    : [p[2] for p in probs],
    "confidence"      : [max(p) for p in probs],
})

# Tambahkan label asli dari folds_v3 (jika ada di folds)
df_train_eval["true_label_id"]   = df_train_eval["filename"].map(fp_to_label)
df_train_eval["true_label_name"] = df_train_eval["true_label_id"].map(CLASS_NAMES)
df_train_eval["is_correct"]      = df_train_eval["true_label_id"] == df_train_eval["pred_label_id"]

# Ringkasan
n_in_folds   = df_train_eval["true_label_id"].notna().sum()
n_correct     = df_train_eval.loc[df_train_eval["true_label_id"].notna(), "is_correct"].sum()
n_wrong       = n_in_folds - n_correct

print(f"Gambar yang ada di folds_v3 : {n_in_folds:,}")
print(f"Prediksi BENAR              : {n_correct:,} ({n_correct/n_in_folds*100:.2f}%)")
print(f"Prediksi SALAH              : {n_wrong:,}  ({n_wrong/n_in_folds*100:.2f}%)")

# Simpan ke Drive
DRIVE_BASE = "/content/drive/MyDrive/BDC2026apace"
out_path   = f"{DRIVE_BASE}/output_trackB/eval_train_mentah_lastlayer_v3.csv"
df_train_eval.to_csv(out_path, index=False)
print(f"\nHasil disimpan: {out_path}")

# Lihat yang salah prediksi
df_wrong = df_train_eval[df_train_eval["is_correct"] == False].sort_values("confidence", ascending=False)
print(f"\nContoh yang masih salah prediksi:")
df_wrong[["filename", "true_label_name", "pred_label_name", "confidence"]].head(10)
